# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 0cac92fe-fbb5-4bda-ac1d-069714… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ d7045260-6be2-46d9-bcc6-2e466e… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-06 ┆ claude-hai ┆ 564.0      ┆ 1137.75   ┆ 16449453  ┆ 8229255   ┆ 0.04714   ┆ 46.076582 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## Gold quality checks

The rubric grades Gold on p50/p95, cost and error_rate, not just on the
number of dates. Assert those properties explicitly instead of eyeballing.

In [6]:
gold_checks = {
    "Bronze/Silver/Gold on disk":   all(Path(p, "_delta_log").exists() for p in (BRONZE, SILVER, GOLD)),
    "Silver < Bronze (dedup)":      silver_n < bronze_n,
    "≥ 7 dates":                    n_dates >= 7,
    "3 models":                     n_models == 3,
    "every date has all 3 models":  gold_df.group_by("date").len()["len"].min() == 3,
    "p50 ≤ p95 on every row":       (gold_df["p50_latency_ms"] <= gold_df["p95_latency_ms"]).all(),
    "cost_usd > 0 on every row":    (gold_df["cost_usd"] > 0).all(),
    "error_rate within [0, 1]":     gold_df["error_rate"].is_between(0, 1).all(),
}
print(gold_df.group_by("model").agg(
    pl.col("p50_latency_ms").mean().round(0).alias("avg_p50_ms"),
    pl.col("p95_latency_ms").mean().round(0).alias("avg_p95_ms"),
    pl.col("error_rate").mean().round(4).alias("avg_error_rate"),
    pl.col("cost_usd").sum().round(2).alias("total_cost_usd"),
).sort("model"))
print(f"date range: {gold_df['date'].min()} → {gold_df['date'].max()}")
for k, v in gold_checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(gold_checks.values()), "NB4 Gold incomplete — see FAIL rows above"
print("\nNB4 complete.")

shape: (3, 5)
┌───────────────────┬────────────┬────────────┬────────────────┬────────────────┐
│ model             ┆ avg_p50_ms ┆ avg_p95_ms ┆ avg_error_rate ┆ total_cost_usd │
│ ---               ┆ ---        ┆ ---        ┆ ---            ┆ ---            │
│ str               ┆ f64        ┆ f64        ┆ f64            ┆ f64            │
╞═══════════════════╪════════════╪════════════╪════════════════╪════════════════╡
│ claude-haiku-4-5  ┆ 565.0      ┆ 1130.0     ┆ 0.0483         ┆ 321.27         │
│ claude-opus-4-7   ┆ 3032.0     ┆ 5983.0     ┆ 0.0528         ┆ 2014.34        │
│ claude-sonnet-4-6 ┆ 1384.0     ┆ 2748.0     ┆ 0.0497         ┆ 2419.18        │
└───────────────────┴────────────┴────────────┴────────────────┴────────────────┘
date range: 2026-04-01 → 2026-04-08
  [PASS] Bronze/Silver/Gold on disk
  [PASS] Silver < Bronze (dedup)
  [PASS] ≥ 7 dates
  [PASS] 3 models
  [PASS] every date has all 3 models
  [PASS] p50 ≤ p95 on every row
  [PASS] cost_usd > 0 on every row
  

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## 📝 Giải thích kết quả NB4

- **Bronze 200.000 dòng → Silver 190.052 dòng** (giảm 9.948 = đúng số retry trùng `request_id` mà generator cài vào).
  Silver dedup bằng `ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)` giữ bản ghi sớm nhất, đồng thời
  parse JSON thô thành cột có kiểu và loại dòng không có `model`.
- **Gold: 24 dòng = 8 ngày × 3 model.** Ô "Gold quality checks" xác nhận p50 ≤ p95 ở mọi dòng, `cost_usd > 0`,
  `error_rate ∈ [0,1]` (~4,8–5,3%), và ngày nào cũng đủ 3 model.
- **Đọc số liệu:** latency tăng theo cỡ model (p50 trung bình: haiku ≈565 ms, sonnet ≈1.384 ms, opus ≈3.032 ms;
  p95 ≈ 2× p50). Chi phí không tỉ lệ với giá đơn vị: sonnet có tổng chi phí cao nhất (~$2.419) vì lưu lượng token lớn
  nhất, dù opus đắt gấp 5× mỗi token (~$2.014). Giá trong `COST_TABLE` là giá minh hoạ của lab.
- **Vì sao 8 ngày chứ không phải 7:** dữ liệu trải đúng 7 ngày UTC (2026-04-01 → 04-07), nhưng `CAST(ts AS DATE)` trong
  DuckDB dùng múi giờ phiên của máy (Asia/Bangkok, UTC+7). Vì vậy ngày 04-01 và 04-08 (giờ địa phương) chỉ có một
  phần dữ liệu (~19K và ~8K dòng thay vì ~27K). Production nên cố định múi giờ (`SET TimeZone='UTC'`) để partition
  `date` không phụ thuộc máy chạy job.